# Titanic EDA

**Dataset:** Kaggle Titanic competition — `data/train.csv`  
**Goal:** Explore survival patterns by gender, passenger class,
and age group using relative frequencies.

## C1 — Setup & Initial Exploration

In [ ]:
# Author:      Wajid Ali Saleem Chaudhry
# Description: Titanic EDA — setup, load data, initial exploration.

# --- Setup ---
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# --- Load Data ---

df = pd.read_csv("data/train.csv")

# --- Initial Exploration ---

print(df.shape)
df.info()

# --- Descriptive Statistics ---

display(df.describe())
display(df.head())

# --- Missing Value Survey ---

missing = df.isnull().sum()
pct = ((missing / len(df)) * 100).round(2)

summary = pd.DataFrame({
    "missing": missing,
    "pct": pct,
})

display(summary[summary["missing"] > 0].sort_values(
    "missing", ascending=False
))

## C2 — Survival by Gender

In [ ]:
# --- C2: Survival by Gender ---
gender_survival = (
    df.groupby("Sex")["Survived"].mean()
      .rename("survival_rate")
)
print(gender_survival.round(3))

fig, ax = plt.subplots(figsize=(6, 4))
gender_survival.plot(kind="bar", ax=ax, rot=0)
ax.set_title("Survival Rate by Gender")
ax.set_ylabel("Survival Rate")
ax.set_xlabel("Gender")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

### Observations
- Females survived at 74.2% vs 18.9% for males — nearly 4x higher.
- Gender is the strongest predictor of survival in this dataset.
- Reflects the "women and children first" evacuation priority.

## C3 — Survival by Passenger Class

In [ ]:
# --- C3: Survival by Passenger Class ---
class_survival = (
    df.groupby("Pclass")["Survived"].mean()
      .rename("survival_rate")
)
print(class_survival.round(3))

fig, ax = plt.subplots(figsize=(6, 4))
class_survival.plot(kind="bar", ax=ax, rot=0)
ax.set_title("Survival Rate by Passenger Class")
ax.set_ylabel("Survival Rate")
ax.set_xlabel("Passenger Class")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

### Observations
- First-class passengers survived at 63.0% vs 24.2% for third-class.
- Survival drops sharply with each class: 63.0% → 47.3% → 24.2%.
- Higher socioeconomic status correlates strongly with survival,
  likely due to cabin location and proximity to lifeboats.

## C4 — Survival by Age Group

In [ ]:
# --- C4a: Age Distribution by Survival (histogram) ---

age_df = df.dropna(subset=["Age"]).copy()

fig, ax = plt.subplots(figsize=(8, 4))

ax.hist(
    age_df.loc[age_df["Survived"] == 1, "Age"],
    bins=20, alpha=0.6, label="Survived",
)
ax.hist(
    age_df.loc[age_df["Survived"] == 0, "Age"],
    bins=20, alpha=0.6, label="Did not survive",
)
ax.set_title("Age Distribution by Survival Status")
ax.set_xlabel("Age")
ax.set_ylabel("Passenger Count")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- C4b: Survival Rate by Age Group (bar chart) ---

bins = [0, 12, 17, 59, 100]
labels = ["Child", "Teen", "Adult", "Senior"]
age_df["age_group"] = pd.cut(
    age_df["Age"], bins=bins, labels=labels
)

age_survival = (
    age_df.groupby("age_group", observed=True)["Survived"]
          .mean()
          .rename("survival_rate")
)
print(age_survival.round(3))

fig, ax = plt.subplots(figsize=(7, 4))
age_survival.plot(kind="bar", ax=ax, rot=0)
ax.set_title("Survival Rate by Age Group")
ax.set_ylabel("Survival Rate")
ax.set_xlabel("Age Group")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

### Observations
- Children (0-12) had the highest survival rate at 58.0%.
- Survival declines steadily with age: 58.0% → 47.7% → 38.6% → 26.9%.
- Seniors (60+) fared worst, consistent with slower evacuation ability.
- 177 rows (19.9%) were excluded due to missing Age values.

## C5 — Summary

Key findings from the Titanic EDA:

- **Gender** was the strongest predictor: females survived at 74.2%
  vs 18.9% for males — nearly 4x higher.
- **Passenger class** showed a clear gradient: 1st (63.0%),
  2nd (47.3%), 3rd (24.2%), reflecting wealth and deck proximity
  to lifeboats.
- **Age** followed evacuation priority: children (58.0%) fared best,
  seniors (26.9%) fared worst.
- **Relative frequencies were essential** — raw counts would mislead
  because cohort sizes differ significantly across all three dimensions.